# 데이터 전처리 코드

**원자료**: 제10차 산업안전보건 실태조사(건설업, 2021) — 1,502개 사업장
**최종 산출물**: 전처리_최종_v3.csv — 1,375개 사업장 x 17개 변수

전처리 과정의 상세 근거는 `전처리_근거_및_과정v4.md` 참고.

## 0. 데이터 로드

In [ ]:
import pandas as pd
import numpy as np

raw = pd.read_csv('/content/제10차_산업안전보건_실태조사_raw_data_건설업_230824.CSV', encoding='cp949')
print(f"원본 데이터: {raw.shape[0]}개 사업장, {raw.shape[1]}개 변수")

## 1. 결측치 및 이상치 제거 (표본 확정)

| 단계 | 제거 대상 | 제거 기준 | 제거 수 | 잔여 |
|:---:|:---|:---|:---:|:---:|
| 시작 | - | 원자료 | - | 1,502 |
| 1 | 종속변수 | Q27_3_1~Q27_3_3 모두 NaN | 16 | 1,486 |
| 2 | 종속변수 이상치 | Q27_3_3=30 (사망 30건 기록, 입력 오류) | 1 | 1,485 |
| 3 | 독립변수 (안전조직) | Q6=9 (무응답) | 21 | 1,464 |
| 4 | 독립변수 (위원회) | Q10=4, Q10=9, Q10_1=9 (무응답/불명) | 62 | 1,402 |
| 5 | 독립변수 (위험성평가) | Q14=NaN (위험요인 없음으로 구조적 미응답) | 24 | 1,378 |
| 6 | 조절변수 (전문지도) | Q9=9 (무응답) | 3 | 1,375 |

In [ ]:
df = raw.copy()
print(f"[시작] {len(df)}개")

# 단계 1: 종속변수 결측 제거 (Q27_3_1~Q27_3_3 모두 NaN)
target_cols = ['Q27_3_1', 'Q27_3_2', 'Q27_3_3']
mask_all_nan = df[target_cols].isna().all(axis=1)
df = df[~mask_all_nan]
print(f"[1단계] 종속변수 NaN 제거 → {len(df)}개 (제거: {mask_all_nan.sum()})")

# 단계 2: 종속변수 이상치 제거 (Q27_3_3=30)
mask_outlier = df['Q27_3_3'] == 30
df = df[~mask_outlier]
print(f"[2단계] Q27_3_3=30 이상치 제거 → {len(df)}개 (제거: {mask_outlier.sum()})")

# 단계 3: Q6=9 (안전조직 무응답) 제거
mask_q6 = df['Q6'] == 9
df = df[~mask_q6]
print(f"[3단계] Q6=9 제거 → {len(df)}개 (제거: {mask_q6.sum()})")

# 단계 4: Q10=4(잘 모르겠음) 또는 Q10=9(무응답) 제거 + Q10_1=9(무응답) 제거
mask_q10 = df['Q10'].isin([4, 9])
df = df[~mask_q10]
mask_q10_1 = df['Q10_1'] == 9
df = df[~mask_q10_1]
print(f"[4단계] Q10/Q10_1 무응답 제거 → {len(df)}개 (제거: {mask_q10.sum() + mask_q10_1.sum()})")

# 단계 5: Q14=NaN (위험성평가 구조적 미응답) 제거
mask_q14 = df['Q14'].isna()
df = df[~mask_q14]
print(f"[5단계] Q14=NaN 제거 → {len(df)}개 (제거: {mask_q14.sum()})")

# 단계 6: Q9=9 (전문지도 무응답) 제거
mask_q9 = df['Q9'] == 9
df = df[~mask_q9]
print(f"[6단계] Q9=9 제거 → {len(df)}개 (제거: {mask_q9.sum()})")

print(f"\n최종 분석 대상: {len(df)}개 사업장")

## 2. 종속변수 구성

**사고발생** (이진변수): Q27_3_1(4~89일) + Q27_3_2(90일 이상) + Q27_3_3(사망) 중 하나라도 1건 이상 발생하면 1, 모두 0이면 0

In [ ]:
# 종속변수: 사고발생
df['사고발생'] = ((df['Q27_3_1'].fillna(0) > 0) |
                  (df['Q27_3_2'].fillna(0) > 0) |
                  (df['Q27_3_3'].fillna(0) > 0)).astype(int)

print("사고발생 분포:")
print(df['사고발생'].value_counts().sort_index())
print(f"사고발생 비율: {df['사고발생'].mean():.3f}")

## 3. 독립변수 전처리

### 독립변수 A: 내부 관리

In [ ]:
# (1) 안전조직수준 (Q6 + Q7_1)
# 0 = 전담부서 없음 + 안전담당 직원 없음 (Q6=2, Q7_1=2)
# 1 = 전담부서 없음 + 안전담당 직원 있음 (Q6=2, Q7_1=1)
# 2 = 전담부서 있음 (Q6=1)
def encode_safety_org(row):
    if row['Q6'] == 1:
        return 2
    elif row['Q6'] == 2:
        if row['Q7_1'] == 1:
            return 1
        else:
            return 0
    return np.nan

df['안전조직수준'] = df.apply(encode_safety_org, axis=1).astype(int)
print("안전조직수준:")
print(df['안전조직수준'].value_counts().sort_index())

In [ ]:
# (2) 위원회수준 (Q10 + Q10_1)
# 0 = 위원회 미운영 (Q10=3)
# 1 = 운영하나 이행관리 미흡 (Q10=1,2 & Q10_1=2,3)
# 2 = 운영 + 지속적 이행관리 (Q10=1,2 & Q10_1=1)
def encode_committee(row):
    if row['Q10'] == 3:
        return 0
    elif row['Q10'] in [1, 2]:
        if row['Q10_1'] == 1:
            return 2
        elif row['Q10_1'] in [2, 3]:
            return 1
        else:
            return 1  # Q10_1 결측이나 기타 → 비관리로 처리
    return np.nan

df['위원회수준'] = df.apply(encode_committee, axis=1).astype(int)
print("위원회수준:")
print(df['위원회수준'].value_counts().sort_index())

In [ ]:
# (3) 인증보유 (Q12_1 + Q12_2)
# KOSHA-MS(Q12_1=1) 또는 ISO45001(Q12_2=1) 중 하나라도 보유하면 1
df['인증보유'] = ((df['Q12_1'] == 1) | (df['Q12_2'] == 1)).astype(int)
print("인증보유:")
print(df['인증보유'].value_counts().sort_index())

### 독립변수 B: 현장 안전 행동

In [ ]:
# (4) 위험성평가수준 (Q14 + Q14_2_2)
# 0 = 미실시 (Q14=1)
# 1 = 실시 + 근로자 미참여 (Q14=2,3 & Q14_2_2=2)
# 2 = 실시 + 근로자 참여 (Q14=2,3 & Q14_2_2=1)
def encode_risk_assess(row):
    if row['Q14'] == 1:
        return 0
    elif row['Q14'] in [2, 3]:
        if row['Q14_2_2'] == 1:
            return 2
        elif row['Q14_2_2'] == 2:
            return 1
        else:
            return 1  # Q14_2_2 결측 → 미참여로 처리
    return np.nan

df['위험성평가수준'] = df.apply(encode_risk_assess, axis=1).astype(int)
print("위험성평가수준:")
print(df['위험성평가수준'].value_counts().sort_index())

In [ ]:
# (5~8) 리커트 변수: 원본 그대로 사용
df['교육훈련도움'] = df['Q16_10']
df['정리정돈상태'] = df['Q16_14']
df['작업중지권']   = df['Q16_17']
df['작업반장기여'] = df['Q17_4']

for name, col in [('교육훈련도움','Q16_10'), ('정리정돈상태','Q16_14'),
                   ('작업중지권','Q16_17'), ('작업반장기여','Q17_4')]:
    print(f"{name} (원본 {col}): 평균={df[name].mean():.2f}, 범위={df[name].min()}~{df[name].max()}")

## 4. 조절변수 전처리 (외부 기관 개입)

In [ ]:
# 전문지도 (Q9): 1=수혜→1, 2=미수혜→0
df['전문지도'] = (df['Q9'] == 1).astype(int)

# 고용노동부감독 (Q30): 1=있음→1, 2=없음→0
df['고용노동부감독'] = (df['Q30'] == 1).astype(int)

# 안전보건공단지원 (Q31): 1=있음→1, 2=없음→0
df['안전보건공단지원'] = (df['Q31'] == 1).astype(int)

for name in ['전문지도', '고용노동부감독', '안전보건공단지원']:
    print(f"{name}: {dict(df[name].value_counts().sort_index())}")

## 5. 통제변수 전처리 (현장 특성)

In [ ]:
# (1) 공사규모 (SQ2): 3+4 합산 → 3
# 1=50~120억, 2=120~800억, 3=800억 이상
df['공사규모'] = df['SQ2'].apply(lambda x: 3 if x >= 3 else x)
print("공사규모:")
print(df['공사규모'].value_counts().sort_index())

In [ ]:
# (2) 발주처 (Q1_4): 1+2→1(정부/공공), 3→2(사기업), 4+5→3(자체/기타)
def encode_client(x):
    if x in [1, 2]:
        return 1
    elif x == 3:
        return 2
    elif x in [4, 5]:
        return 3
    return np.nan

df['발주처'] = df['Q1_4'].apply(encode_client).astype(int)
print("발주처:")
print(df['발주처'].value_counts().sort_index())

In [ ]:
# (3) 기성공정률 (Q1_5): 원본 그대로 사용 (1~6)
df['기성공정률'] = df['Q1_5']
print("기성공정률:")
print(df['기성공정률'].value_counts().sort_index())

In [ ]:
# (4) 공사종류 (Q2): 12개 → 7개 재분류
# 1=아파트, 2=공장, 3=근생/빌딩, 4=학교/병원/건축기타(원본4+5),
# 5=도로/철도/교량(원본6+7+8), 6=상하수도/토목기타(원본9+10), 7=전기통신/기타(원본11+12)
q2_map = {1:1, 2:2, 3:3, 4:4, 5:4, 6:5, 7:5, 8:5, 9:6, 10:6, 11:7, 12:7}
df['공사종류'] = df['Q2'].map(q2_map)
print("공사종류:")
print(df['공사종류'].value_counts().sort_index())

In [ ]:
# (5) 외국인비율 (Q4_3 / Q4_1 x 100)
df['외국인비율'] = (df['Q4_3'] / df['Q4_1'] * 100).round(2)
print(f"외국인비율: 평균={df['외국인비율'].mean():.2f}%, 중위수={df['외국인비율'].median():.2f}%, "
      f"최대={df['외국인비율'].max():.2f}%")
print(f"0%인 사업장: {(df['외국인비율'] == 0).sum()}개 ({(df['외국인비율'] == 0).mean()*100:.1f}%)")

## 6. 최종 데이터 구성 및 검증

In [ ]:
# 최종 17개 변수만 추출
final_cols = [
    # 독립변수 A
    '안전조직수준', '위원회수준', '인증보유',
    # 독립변수 B
    '위험성평가수준', '교육훈련도움', '정리정돈상태', '작업중지권', '작업반장기여',
    # 조절변수
    '전문지도', '고용노동부감독', '안전보건공단지원',
    # 통제변수
    '공사규모', '발주처', '기성공정률', '공사종류', '외국인비율',
    # 종속변수
    '사고발생'
]

df_final = df[final_cols].copy()
print(f"최종 데이터: {df_final.shape[0]}개 사업장 x {df_final.shape[1]}개 변수")
print(f"결측치: {df_final.isnull().sum().sum()}")
print()
print("변수별 dtype:")
print(df_final.dtypes)

In [ ]:
# 기존 전처리 파일과 비교 검증
try:
    df_ref = pd.read_csv('/content/전처리_최종_v3.csv')
    match = df_final.reset_index(drop=True).equals(df_ref.reset_index(drop=True))
    print(f"기존 파일과 일치 여부: {match}")
    if not match:
        # 차이 확인
        for col in final_cols:
            if not df_final[col].reset_index(drop=True).equals(df_ref[col].reset_index(drop=True)):
                diff = (df_final[col].reset_index(drop=True) != df_ref[col].reset_index(drop=True)).sum()
                print(f"  {col}: {diff}건 불일치")
except FileNotFoundError:
    print("기존 전처리 파일이 없어 비교 생략")

In [ ]:
# 기술통계 확인
display(df_final.describe().round(2))

In [ ]:
# CSV 저장
df_final.to_csv('/content/전처리_최종_v3.csv', index=False, encoding='utf-8-sig')
print("저장 완료: 전처리_최종_v3.csv")